# Text CLIP Cosine Similarity Example

This notebook demonstrates text-guided relighting by minimizing cosine distance between the relit image and the target text in the embedding space of the base SigLIP model.

The notebook is configured to support two objectives:
1. `CLIPCosineSimilarity`: Maximizes cosine similarity between the image embedding of the relit image and target prompt embedding (Equation 1 in the [paper](https://diglib.eg.org/items/41c55677-4f59-4f13-8bd5-ecfb98b7f083))
2. `CLIPDirectionalCosineSimilarity`: Measures similarity between the image edit direction and text edit direction (Equation 2 in the [paper](https://diglib.eg.org/items/41c55677-4f59-4f13-8bd5-ecfb98b7f083))

In [ ]:
import os
import sys

if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

from torchvision.transforms.v2 import RandomChoice, RandomResizedCrop

from examples.example_scenes import (
    BlenderManScene,
    CandleScene,
    CarScene,
    CarStudioScene,
    DinoScene,
    EinarScene,
    EinarSmallDomeScene,
    FlowerPotScene,
    HouseScene,
    RedCarScene,
    SciFiRobotScene,
    SpringPortraitScene,
    SpringPortraitSmallDomeScene,
    SpringScene,
)
from losses.clip import CLIPCosineSimilarity, CLIPDirectionalCosineSimilarity
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase
from utils.color.tonemapping.agx_looks import AgXPunchyLook
from utils.model.model_utils import create_clip_model_and_tokenizer
from utils.optimize import optimize_with_criterion


In [ ]:
# Select the scene to optimize (uncomment the desired scene)
scene = SciFiRobotScene(device=device)
# scene = SpringScene(device=device)
# scene = CarScene(device=device)
# scene = BlenderManScene(device=device)
# scene = RedCarScene(device=device)
# scene = CandleScene(device=device)
# scene = HouseScene(device=device)
# scene = DinoScene(device=device)
# scene = FlowerPotScene(device=device)
# scene = CarStudioScene(configuration='dome_lights', device=device)
# scene = EinarScene(device=device)
# scene = EinarSmallDomeScene(device=device)
# scene = SpringPortraitScene(device=device)
# scene = SpringPortraitSmallDomeScene(device=device)


Load the model:

In [ ]:
clip_model_name = "ViT-B-16-SigLIP-512"
clip_pretrained = "webli"
color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())

model, tokenizer, preprocess_eval = create_clip_model_and_tokenizer(
    clip_model_name,
    device=device,
    pretrained=clip_pretrained,
)

In [ ]:
# Text Prompts
# initial_text = "" # To use absolute CLIP cosine similarity, leave initial_text empty
initial_text = "flat, unappealing lighting"
target_text = "dramatic moody cinematic lighting, volumetric rim light"


if initial_text:
    criterion = CLIPDirectionalCosineSimilarity(
        initial_text,
        target_text,
        scene.get_combined_image(color_space_converter).permute(2, 1, 0),
        model,
        tokenizer,
        device=device,
        preprocess=preprocess_eval,
    )
    title_prefix = f"Directional CLIP ('{initial_text}' -> '{target_text}')"
else:
    criterion = CLIPCosineSimilarity(
        target_text,
        model,
        tokenizer,
        device=device,
        preprocess=preprocess_eval,
    )
    title_prefix = f"Standard CLIP ('{target_text}')"

In [ ]:
# Hyperparameters
lr = 0.05
n_iter = 250
global_seed = 2

size = model.visual.preprocess_cfg["size"] or (224, 224) # type: ignore

optimize_with_criterion(
    scene,
    lr,
    n_iter,
    criterion,
    starting_multiplier_std=(0.1, 0.1, 0.1),
    output_subdirectory_name="text_clip_cosine_similarity_example",
    n_results=4,
    augmentation=RandomChoice([RandomResizedCrop(size=size, scale=(0.1, 1.0), antialias=True)]), # type: ignore
    render_color_space_converter=color_space_converter,
    require_physically_plausible_multipliers=True,
    title_prefix=title_prefix,
    device=device,
    save_every=50,
    model_name=clip_model_name,
    pretrained_source=clip_pretrained,
    seed=global_seed,
    show_images_after_augmentation=False,
)
